control prompt: 

Hey — can you help me summarize some sales data? 

We’ve got a CSV file with order-level transactions. I need you to generate python code for me to run locally to get at the following insights: 

1. Total sales grouped by product_code 
2. Total sales grouped by customer_id 
3. Monthly sales totals based on the date column 

You can skip rows where status isn’t "Completed" — we’re only looking at finalized transactions. 

The file is located at C:/data/sales.csv 

and includes these columns: 
- order_number 
- customer_id 
- transaction_amount 
- date 
- product_code 
- status 

Just output everything to the terminal in a clean format — I’ll copy/paste it and run it locally.

In [1]:
# ChatGPT - Free 1/10/2026

import pandas as pd

# Load the CSV
df = pd.read_csv("./data/sales.csv")

# Ensure date column is parsed as datetime
df["date"] = pd.to_datetime(df["date"])

# Filter to completed transactions only
df_completed = df[df["status"] == "Completed"]

# 1. Total sales by product_code
sales_by_product = (
    df_completed
    .groupby("product_code", as_index=False)["transaction_amount"]
    .sum()
    .sort_values("transaction_amount", ascending=False)
)

# 2. Total sales by customer_id
sales_by_customer = (
    df_completed
    .groupby("customer_id", as_index=False)["transaction_amount"]
    .sum()
    .sort_values("transaction_amount", ascending=False)
)

# 3. Monthly sales totals
df_completed["month"] = df_completed["date"].dt.to_period("M")
monthly_sales = (
    df_completed
    .groupby("month", as_index=False)["transaction_amount"]
    .sum()
    .sort_values("month")
)

# Output results
print("\n=== Total Sales by Product Code ===")
print(sales_by_product.to_string(index=False))

print("\n=== Total Sales by Customer ID ===")
print(sales_by_customer.to_string(index=False))

print("\n=== Monthly Sales Totals ===")
print(monthly_sales.to_string(index=False))



=== Total Sales by Product Code ===
product_code  transaction_amount
     PRD-C03             4988.34
     PRD-A12             3646.37
     PRD-B07             3089.68
     PRD-D55             2154.54

=== Total Sales by Customer ID ===
customer_id  transaction_amount
   CUST-207             1579.86
   CUST-218             1403.14
   CUST-211             1398.43
   CUST-220             1309.75
   CUST-210             1186.32
   CUST-215              940.89
   CUST-204              869.30
   CUST-201              863.65
   CUST-209              852.86
   CUST-213              823.42
   CUST-216              773.33
   CUST-217              528.30
   CUST-202              463.99
   CUST-203              430.94
   CUST-208              395.47
   CUST-214               38.75
   CUST-205               20.53

=== Monthly Sales Totals ===
  month  transaction_amount
2023-11             4913.47
2023-12             8965.46


In [3]:
import pandas as pd
import time
from datetime import datetime


def summarize_sales_data(file_path: str = "./data/sales.csv", debug: bool = True) -> None:
    """
    Summarize sales performance by product, customer, and month.

    This function reads a CSV file of sales transactions, filters for completed
    orders, and calculates three key summaries:
        1. Total sales by product_code
        2. Total sales by customer_id
        3. Monthly total sales based on the transaction date

    The results are printed cleanly to the terminal.

    Parameters
    ----------
    file_path : str, optional
        Path to the sales CSV file. Default is './data/sales.csv'.
    debug : bool, optional
        If True, prints detailed debug logs. Default is True.

    Returns
    -------
    None
        The function prints summaries to stdout and does not return a value.

    Raises
    ------
    FileNotFoundError
        If the CSV file cannot be located at the specified path.
    ValueError
        If required columns are missing from the CSV file.
    """

    start_time = time.time()
    if debug:
        print(f"[{datetime.now()}] Starting sales summary process...")
        print(f"Attempting to load data from: {file_path}")

    # Load CSV
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Could not find the file at {file_path}") from e

    # Validate expected columns
    required_cols = {
        "order_number",
        "customer_id",
        "transaction_amount",
        "date",
        "product_code",
        "status",
    }

    if not required_cols.issubset(df.columns):
        missing = required_cols - set(df.columns)
        raise ValueError(f"The following required columns are missing: {missing}")

    if debug:
        print(f"✅ Successfully loaded {len(df)} rows.")
        print("Filtering to completed transactions only...")

    # Filter for completed transactions
    df = df[df["status"].str.lower() == "completed"]

    if debug:
        print(f"Remaining rows after filtering: {len(df)}")

    # Convert date column to datetime
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Drop rows with invalid or missing dates
    df = df.dropna(subset=["date"])

    # --- Summaries ---
    if debug:
        print("Computing total sales by product_code...")
    sales_by_product = df.groupby("product_code")["transaction_amount"].sum().sort_values(ascending=False)

    if debug:
        print("Computing total sales by customer_id...")
    sales_by_customer = df.groupby("customer_id")["transaction_amount"].sum().sort_values(ascending=False)

    if debug:
        print("Computing monthly total sales...")
    df["month"] = df["date"].dt.to_period("M")
    monthly_sales = df.groupby("month")["transaction_amount"].sum().sort_index()

    # --- Output ---
    print("\n" + "=" * 60)
    print("📦 TOTAL SALES BY PRODUCT CODE")
    print("=" * 60)
    print(sales_by_product.to_string())

    print("\n" + "=" * 60)
    print("👤 TOTAL SALES BY CUSTOMER ID")
    print("=" * 60)
    print(sales_by_customer.to_string())

    print("\n" + "=" * 60)
    print("🗓️  MONTHLY TOTAL SALES")
    print("=" * 60)
    print(monthly_sales.to_string())

    runtime = time.time() - start_time
    print(f"\n✅ Sales summary completed in {runtime:.2f} seconds.\n")


# --- Optional: Run directly from terminal ---
if __name__ == "__main__":
    summarize_sales_data()


[2026-01-10 13:11:42.152650] Starting sales summary process...
Attempting to load data from: ./data/sales.csv
✅ Successfully loaded 50 rows.
Filtering to completed transactions only...
Remaining rows after filtering: 50
Computing total sales by product_code...
Computing total sales by customer_id...
Computing monthly total sales...

📦 TOTAL SALES BY PRODUCT CODE
product_code
PRD-C03    4988.34
PRD-A12    3646.37
PRD-B07    3089.68
PRD-D55    2154.54

👤 TOTAL SALES BY CUSTOMER ID
customer_id
CUST-207    1579.86
CUST-218    1403.14
CUST-211    1398.43
CUST-220    1309.75
CUST-210    1186.32
CUST-215     940.89
CUST-204     869.30
CUST-201     863.65
CUST-209     852.86
CUST-213     823.42
CUST-216     773.33
CUST-217     528.30
CUST-202     463.99
CUST-203     430.94
CUST-208     395.47
CUST-214      38.75
CUST-205      20.53

🗓️  MONTHLY TOTAL SALES
month
2023-11    4913.47
2023-12    8965.46
Freq: M

✅ Sales summary completed in 0.01 seconds.

